# WorldKit in 5 Minutes

Train, encode, predict, and plan with JEPA world models.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DilpreetBansi/worldkit/blob/main/notebooks/01_quickstart.ipynb)

In [ ]:
!pip install worldkit -q

## 1. Load a Pre-trained Model

WorldKit hosts pre-trained JEPA world models on Hugging Face Hub.
Load one in a single line.

In [ ]:
from worldkit import WorldModel
import numpy as np
import matplotlib.pyplot as plt

model = WorldModel.from_hub("DilpreetBansi/pusht-base", device="cpu")

print(f"Model:      {model.config.name}")
print(f"Latent dim: {model.latent_dim}")
print(f"Parameters: {model.num_params / 1e6:.1f}M")

## 2. Encode Observations

The encoder maps a raw RGB frame to a compact latent vector.
Input: any `(H, W, 3)` numpy array. Output: a latent vector of size `latent_dim`.

In [ ]:
# Create a synthetic 96x96 RGB frame (replace with a real observation)
frame = np.random.randint(0, 255, (96, 96, 3), dtype=np.uint8)

z = model.encode(frame)

print(f"Latent shape: {z.shape}")
print(f"Norm:         {np.linalg.norm(z):.4f}")
print(f"Min / Max:    {z.min():.4f} / {z.max():.4f}")

# Visualize the latent vector
fig, ax = plt.subplots(figsize=(10, 2))
ax.bar(range(len(z)), z, width=1.0, color="steelblue")
ax.set_xlabel("Latent dimension")
ax.set_ylabel("Value")
ax.set_title("Encoded latent vector")
plt.tight_layout()
plt.show()

## 3. Predict Future States

Given an observation and a sequence of actions, the model predicts
future latent states without decoding back to pixels.
This is the core of a world model — imagining what happens next.

In [ ]:
actions = [(0.5, 0.0)] * 5
result = model.predict(frame, actions=actions)

print(f"Trajectory shape: {result.latent_trajectory.shape}")
print(f"Confidence:       {result.confidence:.4f}")
print(f"Steps predicted:  {result.steps}")

# Latent norm at each predicted step
norms = [np.linalg.norm(result.latent_trajectory[i]) for i in range(result.steps)]

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, len(norms) + 1), norms, marker="o", color="steelblue")
ax.set_xlabel("Step")
ax.set_ylabel("Latent norm")
ax.set_title("Predicted latent norms over time")
plt.tight_layout()
plt.show()

## 4. Plan Actions

Given a current observation and a goal observation, the CEM planner
searches for an action sequence that transitions between them —
no reward function needed.

In [ ]:
current_frame = np.random.randint(0, 255, (96, 96, 3), dtype=np.uint8)
goal_frame = np.random.randint(0, 255, (96, 96, 3), dtype=np.uint8)

plan = model.plan(current_frame, goal_frame, max_steps=20)

print(f"Planned steps:        {len(plan.actions)}")
print(f"Expected cost:        {plan.expected_cost:.4f}")
print(f"Success probability:  {plan.success_probability:.4f}")
print(f"Planning time:        {plan.planning_time_ms:.1f} ms")

# Plot the planned action sequence
actions_arr = np.array(plan.actions)
fig, ax = plt.subplots(figsize=(8, 3))
steps = range(1, len(actions_arr) + 1)
ax.plot(steps, actions_arr[:, 0], marker="o", label="Action dim 0", color="steelblue")
ax.plot(steps, actions_arr[:, 1], marker="s", label="Action dim 1", color="coral")
ax.set_xlabel("Step")
ax.set_ylabel("Action value")
ax.set_title("Planned action sequence")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Plausibility Scoring

Score how physically plausible a sequence of observations is.
Coherent sequences score high; random sequences score low.
This is useful for anomaly detection and data quality filtering.

In [ ]:
# Coherent sequence: small perturbations of a base frame
base = np.random.randint(100, 150, (96, 96, 3), dtype=np.uint8)
coherent = [
    np.clip(base + np.random.randint(-5, 6, base.shape), 0, 255).astype(np.uint8)
    for _ in range(5)
]

# Random sequence: completely independent frames
random_seq = [
    np.random.randint(0, 255, (96, 96, 3), dtype=np.uint8)
    for _ in range(5)
]

score_coherent = model.plausibility(coherent)
score_random = model.plausibility(random_seq)

print(f"Coherent sequence score: {score_coherent:.4f}")
print(f"Random sequence score:   {score_random:.4f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Coherent", "Random"], [score_coherent, score_random],
       color=["steelblue", "coral"])
ax.set_ylabel("Plausibility score")
ax.set_title("Physical plausibility comparison")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 6. Works Across Environments

WorldKit isn't tied to one task. The same API works on any environment
that produces pixel observations. Here's CartPole — a completely different
domain — with zero code changes.

In [ ]:
cartpole = WorldModel.from_hub("DilpreetBansi/cartpole-base", device="cpu")

print(f"Model:      {cartpole.config.name}")
print(f"Latent dim: {cartpole.latent_dim}")
print(f"Parameters: {cartpole.num_params / 1e6:.1f}M")

# Encode a CartPole frame
cartpole_frame = np.random.randint(0, 255, (96, 96, 3), dtype=np.uint8)
z_cart = cartpole.encode(cartpole_frame)
print(f"\nCartPole latent shape: {z_cart.shape}")
print(f"CartPole latent norm:  {np.linalg.norm(z_cart):.4f}")

# Predict future states (CartPole has 1D action: left/right)
cart_result = cartpole.predict(cartpole_frame, actions=[(1.0,)] * 5)
print(f"Predicted {cart_result.steps} steps ahead")

## 7. Train From Scratch

Prepare an HDF5 dataset with `pixels` and `actions`, then train
a model from scratch. The `nano` config trains in seconds on Colab.

In [ ]:
import h5py

# Create synthetic data: 50 episodes, 16 timesteps each
# Shape: (episodes, timesteps, height, width, channels)
pixels = np.random.randint(0, 255, (50, 16, 96, 96, 3), dtype=np.uint8)
actions = np.random.randn(50, 16, 2).astype(np.float32)

with h5py.File("my_data.h5", "w") as f:
    f.create_dataset("pixels", data=pixels, compression="gzip")
    f.create_dataset("actions", data=actions, compression="gzip")

print("Dataset saved: my_data.h5")

In [ ]:
# Train a nano model (fast on CPU / Colab)
trained = WorldModel.train(data="my_data.h5", config="nano", epochs=5)

print(f"\nTrained {trained.num_params:,} parameters")
print(f"Latent dim: {trained.latent_dim}")

In [ ]:
# Save and reload
trained.save("my_model.wk")
reloaded = WorldModel.load("my_model.wk")
print(f"Saved and reloaded: {reloaded.num_params:,} params")

## Next Steps

You've seen the core WorldKit workflow. Explore further:

| Notebook | What you'll learn |
|----------|-------------------|
| [02 — Train on Custom Env](02_train_custom_env.ipynb) | Record CartPole data, train, predict, and plan |
| [03 — Anomaly Detection](03_anomaly_detection.ipynb) | Detect physically implausible events |
| [04 — Latent Probing](04_latent_probing.ipynb) | What has the model actually learned? |
| [05 — Model Comparison](05_model_comparison.ipynb) | Compare nano vs base vs large |
| [06 — Export & Deploy](06_export_deploy.ipynb) | Export to ONNX, benchmark inference |
| [07 — Plan Robot Actions](07_plan_robot_actions.ipynb) | Goal-conditioned and hierarchical planning |

**Links:**
- [GitHub](https://github.com/DilpreetBansi/worldkit)
- [Documentation](https://worldkit.readthedocs.io)
- [Pre-trained models on HuggingFace](https://huggingface.co/DilpreetBansi)